# Ella2027 — LTX-2.5 22B on Kaggle T4

A reproducible Kaggle notebook for **LTX-2.5 22B Distilled NVFP4 + Gemma 4 12B INT8 ConvRot**, using Wan2GP.

This notebook connects WanGP directly to the Ella2027 job queue through WanGP's supported Python API. It does not rely on Gradio's temporary public URL. Existing checkpoints under `/tmp/ella-ltx25` are reused when they are still present.

**Initial profile:** 768×448, up to 5 seconds per clip, batch size 1. Kaggle T4 does not provide native optimized NVFP4 kernels; performance and successful completion depend on the active Kaggle runtime and WanGP compatibility.

## 1. Preflight — GPU, internet, and storage
Run this first. It does not download model weights.

In [ ]:
import os, shutil, torch, subprocess, sys
print('Python:', sys.version)
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Enable a Kaggle GPU accelerator before continuing.')
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f'GPU {i}: {p.name} | VRAM {p.total_memory/1024**3:.2f} GB')
MODEL_DIR = '/tmp/ella-ltx25'
os.makedirs(MODEL_DIR, exist_ok=True)
for path in ['/tmp', '/kaggle/working']:
    u = shutil.disk_usage(path)
    print(f'{path}: free={u.free/1024**3:.1f} GB | total={u.total/1024**3:.1f} GB')
print('Model directory:', MODEL_DIR)
print('Preflight complete.')

## 2. Get Wan2GP source
The repository is cloned into the small working area. Model weights are stored separately in `/tmp`.

In [ ]:
import os, subprocess
WGP_DIR = '/kaggle/working/Wan2GP'
if not os.path.isdir(os.path.join(WGP_DIR, '.git')):
    if os.path.exists(WGP_DIR):
        raise RuntimeError(f'{WGP_DIR} exists but is not a Git checkout. Rename/remove it before continuing.')
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/deepbeepmeep/Wan2GP.git', WGP_DIR], check=True)
else:
    print('Wan2GP checkout already exists; keeping it unchanged.')
os.chdir(WGP_DIR)
subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], check=True)
print('Wan2GP source ready:', WGP_DIR)

## 3. Install dependencies without replacing Kaggle's CUDA PyTorch
Pin the already-installed Torch family during requirements installation. This avoids an unnecessary large Torch/CUDA replacement.

In [ ]:
import os, sys, subprocess, torch
import importlib.util
from pathlib import Path
os.chdir(WGP_DIR)
constraints = Path('/tmp/ella-kaggle-constraints.txt')
pins = [f'torch=={torch.__version__.split("+")[0]}']
for package in ['torchvision', 'torchaudio']:
    if importlib.util.find_spec(package) is not None:
        mod = __import__(package)
        pins.append(f'{package}=={mod.__version__.split("+")[0]}')
constraints.write_text('\n'.join(pins) + '\n')
print('Constraints:', constraints.read_text())
if not Path('requirements.txt').is_file():
    raise FileNotFoundError('Wan2GP requirements.txt was not found.')
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '-c', str(constraints), '-r', 'requirements.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', 'huggingface_hub>=0.30.0'], check=True)
print('Dependency installation complete. If Kaggle asks for a runtime restart, restart once and rerun cells from section 1.')

## 4. Download the two required LTX-2.5 checkpoints
Downloads are resumable through Hugging Face Hub's cache. If a download is interrupted, rerun this cell; completed files are reused. Do not save these large files in `/kaggle/working`.

In [ ]:
import os
from pathlib import Path
from huggingface_hub import hf_hub_download
MODEL_DIR = '/tmp/ella-ltx25'
os.makedirs(MODEL_DIR, exist_ok=True)
main_name = 'ltx-2.5-22b-distilled_diffusion_model_nvfp4.safetensors'
encoder_name = 'gemma4-12b-ltx-v1_int8_convrot.safetensors'
print('Downloading main LTX-2.5 checkpoint (large file; resumable)...')
main_path = hf_hub_download(repo_id='DeepBeepMeep/LTX-2', filename=main_name, local_dir=MODEL_DIR)
print('Main checkpoint ready:', main_path)
print('Downloading Gemma 4 12B INT8 ConvRot text encoder...')
encoder_path = hf_hub_download(repo_id='DeepBeepMeep/LTX-2', filename='gemma4-12b-ltx-v1/' + encoder_name, local_dir=MODEL_DIR)
print('Text encoder ready:', encoder_path)
for p in [Path(main_path), Path(encoder_path)]:
    if not p.is_file() or p.stat().st_size < 100_000_000:
        raise RuntimeError(f'Checkpoint missing or unexpectedly small: {p}')
    print(f'VERIFIED {p.name}: {p.stat().st_size/1024**3:.2f} GB')
print('\nBoth required checkpoints are present.')

## 5. Start the Ella2027-connected WanGP worker

This starts WanGP's in-process Python API, keeps the model session warm, polls Ella2027 for video jobs, and sends completed MP4 files to the app's Cloudinary storage. Keep this cell running while you use Ella2027.

The worker uses the existing Kaggle secret `ELLA_KAGGLE_WORKER_TOKEN` and the Render app URL `https://ella2027.onrender.com`.

In [ ]:
import os, sys, subprocess, requests
from pathlib import Path
from kaggle_secrets import UserSecretsClient

WORKER_URL = "https://raw.githubusercontent.com/WaelNameh84/Ella2027/feature/ltx25-wangp-api-worker/services/kaggle-ltx-worker/wangp_worker.py"
WORKER_PATH = Path("/kaggle/working/wangp_worker.py")
response = requests.get(WORKER_URL, timeout=60)
response.raise_for_status()
WORKER_PATH.write_text(response.text, encoding="utf-8")

secrets = UserSecretsClient()
os.environ["ELLA_KAGGLE_WORKER_TOKEN"] = secrets.get_secret("ELLA_KAGGLE_WORKER_TOKEN")
os.environ["ELLA_API_BASE_URL"] = "https://ella2027.onrender.com"
os.environ["ELLA_WANGP_MODEL_TYPE"] = "ltx2_22B_distilled"

if not os.environ["ELLA_KAGGLE_WORKER_TOKEN"].strip():
    raise RuntimeError("Kaggle secret ELLA_KAGGLE_WORKER_TOKEN is empty.")

print("WanGP Python API worker downloaded.")
print("Ella2027 API:", os.environ["ELLA_API_BASE_URL"])
print("Model type:", os.environ["ELLA_WANGP_MODEL_TYPE"])
print("Starting connected worker. Keep this cell running.")
subprocess.run([sys.executable, str(WORKER_PATH)], check=True)

## 6. First test from Ella2027

1. Keep the worker cell running.
2. In Ella2027, open a scene and choose the **Ella Local GPU Worker** video provider.
3. Select the LTX-2.5 model and start with a scene duration of **5 seconds or less**.
4. The worker claims the job, generates through WanGP, uploads the MP4 to Cloudinary, and saves the result back to that scene.

The initial T4 profile rejects requests longer than five seconds rather than silently shortening them. Audio output depends on the LTX-2.5 settings returned by the installed WanGP model defaults; the worker logs the active audio prompt setting.

## Storage and recovery notes

- `/tmp/ella-ltx25` is temporary and is cleared when Kaggle replaces the runtime. Keep a private Kaggle Dataset copy of the model files if you want to avoid downloading them again.
- `/kaggle/working` contains the small Wan2GP source and worker script; model weights stay outside it.
- The worker uses WanGP's documented `shared.api` Python API and the existing Ella2027 `/api/kaggle-worker` lease/heartbeat/complete/fail endpoints.
- The worker requires the Render secret `KAGGLE_LTX_WORKER_TOKEN` and the matching Kaggle secret `ELLA_KAGGLE_WORKER_TOKEN`.
- The model type used by the worker is `ltx2_22B_distilled`; the worker prints WanGP's model availability at startup. If the checkpoint paths are not configured in the existing WanGP config, fix that configuration before sending jobs.
- The worker keeps polling while this cell runs. Stopping the Kaggle session stops generation availability.
- Kaggle T4 NVFP4 execution may use a slower fallback. No generation-time guarantee is made.